<a href="https://colab.research.google.com/github/SlavaMarina/ib-cs-ml-course/blob/main/week2_supervised/Week2_Lesson3_Lecture.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 IB CS — Неделя 2 · Урок 3 (Лекция)
## Классические алгоритмы, метрики + добор A4.2.2/A4.2.3
### A4.2.2, A4.2.3, A4.3.1, A4.3.2, A4.3.3 (HL Focus)

> ⏱ Длительность: 2 академических часа.
> 📘 Источники: Baumgarten (Hodder IBDP) + MacKenty & Stephenson (Oxford 2025).

---

### 🎯 Что покрываем на этом уроке (по syllabus):

| Syllabus statement | Тема | Command term |
|---|---|---|
| **A4.2.2** | Feature selection | *Describe* |
| **A4.2.3** | Dimensionality reduction | *Describe* |
| **A4.3.1** | Linear regression | *Explain* |
| **A4.3.2** | KNN + Decision Trees | *Explain* |
| **A4.3.3** | Hyperparameter tuning | *Explain* |
| **+** | Metrics (Accuracy/Precision/Recall/F1) | (часть A4.3.3) |
| **+** | Overfitting / Underfitting | (часть A4.3.3) |

---

### ⚠️ Важно перед стартом

> На прошлой неделе мы прошли **A4.2.1** (data cleaning). Сегодня мы **завершаем модуль A4.2 (Data Preprocessing — HL)**, пройдя **A4.2.2** и **A4.2.3** в начале. Затем переходим к **A4.3 (ML Approaches — HL)**.
>
> 💎 Это **половина всего ML-контента IB**. Без чёткого понимания этих тем 6+ не получить.


## 📌 Часть 1 · A4.2.2 Feature Selection (HL)

> **Definition (выучить!):** *Feature selection refers to taking care to select only the most relevant features for use in your machine learning models.*
>
> **Feature:** *a numeric property (variable) that can be used as input for a machine learning algorithm.*

### Зачем это нужно?

Представьте, что у вас 500 признаков на 100 объектов. Это:
- 🐌 **Медленно** — модель обучается часами вместо минут
- 🎯 **Неточно** — модель учит "шум" и переобучается (overfitting)
- 🤯 **Сложно интерпретировать** — невозможно понять, что важно

> 🚨 **Common mistake (Baumgarten, прямая цитата):**
> *"Don't underestimate the importance of feature selection and engineering. Good features are often more important than the choice of model itself."*

### 🔧 Три метода feature selection (по syllabus):

| Метод | Как работает | Когда использовать |
|---|---|---|
| **Filter methods** | статистическая метрика (например, **Pearson's r**) ранжирует фичи независимо от модели | большие датасеты, быстрая первичная фильтрация |
| **Wrapper methods** | перебор подмножеств фич, обучение модели на каждом, выбор лучшего | малые датасеты, нужна точность |
| **Embedded methods** | feature selection встроен в обучение (например, regularization в Lasso) | компромисс между filter и wrapper |

> 💎 **СЕКРЕТ #1:** на экзамене *"Distinguish between filter and wrapper methods"* — главное различие в **computational cost**:
> - **Filter** — дёшево (нет обучения модели)
> - **Wrapper** — дорого (тренируем модель N раз)
> - **Embedded** — баланс (обучение модели + feature selection одновременно)


In [ ]:
# Демонстрация Filter method на простом датасете
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

# Создаём датасет: предсказываем цену дома
n = 200
df = pd.DataFrame({
    'area_m2':      np.random.normal(120, 40, n),
    'bedrooms':     np.random.choice([1,2,3,4,5], n),
    'random_noise': np.random.normal(0, 1, n),       # бесполезная фича!
    'age_years':    np.random.exponential(15, n),
    'temperature':  np.random.normal(20, 5, n),      # ещё одна бесполезная (нет связи с ценой)
})
df['price'] = (df['area_m2'] * 3.5 + df['bedrooms']*25 - df['age_years']*1.2
               + np.random.normal(0, 20, n))

# Pearson's r — filter method
correlations = df.corr()['price'].drop('price').abs().sort_values(ascending=False)
print("=== Pearson |r| с целевой переменной 'price' ===")
print(correlations.round(3))
print("\n💡 Filter method скажет: оставляем area_m2, bedrooms, age_years")
print("    Отбрасываем random_noise и temperature — у них |r| близок к 0")

In [ ]:
# Визуализация: heatmap корреляций
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title('Корреляционная матрица — основа filter method', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

> ⚠️ **Подводный камень (Baumgarten, p. 28):**
> *"As the Spurious Correlations website demonstrates, if you compare enough unrelated data sets, you will find correlations that are, in fact, not real."*
>
> На IB экзамене это бонус: упомяните, что корреляция ≠ причинно-следственная связь. Filter methods могут вводить в заблуждение.

### 🎯 Mini-вопрос для класса
> *"Outline ONE limitation of filter methods compared to wrapper methods."* **[2]**
>
> 💡 Подсказка: filter methods не учитывают **interaction between features**. Если фичи A и B по отдельности слабые, но вместе сильные — filter выбросит обе.


## 📌 Часть 2 · A4.2.3 Dimensionality Reduction (HL)

> **Definition (выучить!):** *Dimensionality reduction is a group of techniques that reduce the number of variables/features in a data set while preserving as much relevant information as possible.*

### ⚠️ Curse of Dimensionality (проклятие размерности)

> 📘 **Прямая цитата из MacKenty/Stephenson:**
> *"In high-dimensional space, the feature space grows exponentially. With so many dimensions, each data point becomes an isolated point, far from others. Essentially, every observation appears unique, with few obvious 'near neighbours'."*

**Что происходит при росте числа признаков:**

| Размерность | Эффект |
|---|---|
| 2–3 | Все ОК, можно визуализировать |
| 10–50 | Точки начинают "разлетаться" |
| 100+ | Все точки одинаково далеки друг от друга — модель не находит паттернов |
| 1000+ | Требуется **экспоненциально** больше данных |

> 💎 **СЕКРЕТ #2:** на вопрос *"Explain the curse of dimensionality"* — обязательно упомяните:
> 1. Exponential growth of feature space
> 2. Sparsity of data points
> 3. Distance-based algorithms (как **KNN**) перестают работать
> 4. Нужно **экспоненциально** больше данных


In [ ]:
# Визуализация curse of dimensionality
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
np.random.seed(0)

for ax, dim in zip(axes, [2, 10, 100]):
    # Сгенерируем 50 точек в N-мерном пространстве
    points = np.random.uniform(0, 1, (50, dim))
    # Посчитаем все попарные расстояния
    from scipy.spatial.distance import pdist
    distances = pdist(points)
    ax.hist(distances, bins=20, color='steelblue', edgecolor='black')
    ax.set_title(f'Размерность = {dim}\nmin={distances.min():.2f}, max={distances.max():.2f}\n'
                 f'ratio max/min = {distances.max()/distances.min():.1f}',
                 fontsize=11)
    ax.set_xlabel('Расстояние между точками')
    ax.set_ylabel('Частота')

plt.suptitle('Curse of Dimensionality: с ростом dim расстояния становятся одинаковыми',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

> 🔬 **Наблюдение:** в 100-мерном пространстве **все точки равноудалены** друг от друга. KNN перестаёт работать, потому что "ближайший сосед" теряет смысл.

### 🛠 Две техники dimensionality reduction

| Техника | Что делает | Сохраняет |
|---|---|---|
| **Feature selection** | удаляет фичи | оригинальный смысл фич |
| **Feature extraction** (например, **PCA**) | создаёт новые фичи как комбинации старых | максимум variance |

> ⚠️ **Важно для IB:** *"Statistical techniques such as PCA and LDA are beyond the scope of this course."* (MacKenty/Stephenson p. 262)
>
> 💎 То есть: **знать названия PCA и LDA нужно** (могут спросить *Identify*), но **выводить формулы — не нужно**. Просто понимать, что они **уменьшают** размерность.

### 📊 Выгоды dimensionality reduction (для IB ответов):

1. **Лучшая визуализация** — можно построить 2D/3D графики
2. **Меньше overfitting** — модель не учит шум
3. **Меньше времени и памяти** — обучение быстрее
4. **Проще анализ** — паттерны легче увидеть

> 💎 **СЕКРЕТ #3 (Baumgarten common mistake):**
> *"Dimensionality reduction does not ALWAYS lead to better model performance. The primary goal is to simplify the model, which can help but might also lead to LOSS of critical information."*
>
> На *Discuss* вопросах **всегда** упоминайте trade-off: simplicity vs information loss.


## 📈 Часть 3 · A4.3.1 Linear Regression

> **Definition:** *Linear regression is a machine learning algorithm that seeks a linear line of best fit for a given data set, from which extrapolations can be made.*

### Формула (нужно знать!):

$$Y = \beta_0 + \beta_1 X + \epsilon$$

| Символ | Смысл |
|---|---|
| $Y$ | dependent variable (что предсказываем) |
| $X$ | independent variable (на основе чего) |
| $\beta_0$ | **intercept** — значение Y при X=0 |
| $\beta_1$ | **slope** — на сколько меняется Y при +1 к X |
| $\epsilon$ | ошибка (error term) |

> 💎 **СЕКРЕТ #4:** часто спрашивают *"Describe the significance of slope and intercept"* **[4]**.
> Полный ответ:
> - **Intercept** — стартовая точка / базовое значение
> - **Slope** — насколько Y меняется при изменении X (положительный = прямая связь)
> - **Slope величина** — сила влияния
> - **Пример конкретный** — например, "intercept = базовая цена дома, slope = доплата за каждый м²"

### Когда использовать?

| Подходит | Не подходит |
|---|---|
| Линейная связь между X и Y | Нелинейные паттерны (используйте polynomial regression) |
| Числовой выход (continuous) | Категориальный выход (тогда classification) |
| Понятная интерпретация важна | Сложные паттерны (тогда нейросети) |


In [ ]:
# Демо: linear regression на ценах домов
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Данные: площадь дома → цена
np.random.seed(0)
area = np.random.uniform(50, 300, 50)
price = 178220 + 1603 * area + np.random.normal(0, 30000, 50)

X = area.reshape(-1, 1)
y = price

model = LinearRegression()
model.fit(X, y)

slope = model.coef_[0]
intercept = model.intercept_
r2 = r2_score(y, model.predict(X))

print(f"Intercept (β₀) = {intercept:,.0f}  → стартовая цена при 0 м²")
print(f"Slope (β₁)     = {slope:,.0f}  → доплата за каждый м²")
print(f"R² (R-squared) = {r2:.3f}  → доля объяснённой дисперсии")

# График
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(area, price, alpha=0.7, color='steelblue', label='Реальные данные')
x_line = np.linspace(50, 300, 100)
y_line = intercept + slope * x_line
ax.plot(x_line, y_line, 'r-', linewidth=2, label=f'Y = {intercept:,.0f} + {slope:,.0f}·X')
ax.set_xlabel('Площадь (м²)')
ax.set_ylabel('Цена ($)')
ax.set_title(f'Linear Regression: предсказание цены дома (R²={r2:.3f})',
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 📊 R² (R-squared) — главная метрика для регрессии

> **Definition:** *R-squared is a statistical measure that indicates how well the linear regression model fits the data points.*

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$$

| Значение R² | Интерпретация |
|---|---|
| **0** | модель **не объясняет** изменчивость данных |
| **0.5** | объясняет 50% изменчивости (умеренно) |
| **0.8+** | хорошая модель |
| **1.0** | **идеальное** объяснение (часто = overfitting) |

> 💎 **СЕКРЕТ #5:** R² слишком близкий к 1.0 — это **подозрительно**, обычно overfitting. На экзамене на *"How would you assess accuracy of a regression model?"* — упомяните R² **И ТАКЖЕ** MSE, MAE, или mean absolute percentage error.


## 🎯 Часть 4 · A4.3.2 Classification — KNN и Decision Trees

> **Definition:** *Classification techniques are where a machine learning model has been trained to identify, from a predefined list of categories, which category the input data would most likely belong to.*

**Два главных не-нейросетевых классификатора:**

### 1️⃣ K-Nearest Neighbours (KNN)

> **Definition:** *Data points are categorized based on the categories of the nearest points around them; k is a variable representing how many nearest points should "vote".*

**Как работает (3 шага по MacKenty/Stephenson):**
1. Для новой точки рассчитать **расстояние** до всех известных точек
2. Выбрать **K ближайших** соседей
3. **Голосование большинством** — присваиваем класс большинства

**Свойства KNN:**
- ✅ **Non-parametric** — не предполагает формы данных
- ✅ **Instance-based / Lazy learning** — не строит модель, просто запоминает данные
- ❌ **Чувствителен к шкале** — обязательно **normalize** перед использованием
- ❌ **Медленный на predict** — считает расстояния до всех точек

> ⚠️ **Top tip from Baumgarten:** *"Choose an ODD number for k when the number of classes is even to avoid tie situations."*


In [ ]:
# Визуализация: как K влияет на decision boundary
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_blobs

X_demo, y_demo = make_blobs(n_samples=150, centers=2, cluster_std=1.5, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, k in zip(axes, [1, 5, 25]):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_demo, y_demo)

    # Сетка для visualization
    xx, yy = np.meshgrid(np.linspace(X_demo[:,0].min()-1, X_demo[:,0].max()+1, 200),
                         np.linspace(X_demo[:,1].min()-1, X_demo[:,1].max()+1, 200))
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X_demo[:,0], X_demo[:,1], c=y_demo, cmap='coolwarm',
               edgecolor='k', s=50)
    ax.set_title(f'KNN, k={k}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Признак 1'); ax.set_ylabel('Признак 2')

plt.suptitle('Эффект K: маленький K → переобучение · большой K → недообучение',
             fontsize=13, y=1.03)
plt.tight_layout(); plt.show()

> 🔬 **Наблюдение:**
> - **k=1**: граница "извилистая" — модель **переобучается** (overfitting)
> - **k=5**: оптимально для этого случая
> - **k=25**: граница "сглажена" — модель **недообучается** (underfitting)
>
> 💎 **СЕКРЕТ #6:** *"Outline how choice of k affects classification accuracy"* — стандартный 2-балльный вопрос. Ответ должен включать **оба** края: маленькое k → overfit; большое k → underfit.

---

### 2️⃣ Decision Trees

> **Definition:** *A graphical representation of conditions that result in a classification decision; think of it as a decision-making flowchart that the ML model creates automatically.*

**Как работает (по MacKenty/Stephenson):**
1. **Root node** — начальный узел
2. **Split** — данные делятся по фиче, дающей наиболее однородные подмножества (по **Gini impurity** или **entropy**)
3. **Recursive splitting** — повторяем до termination condition
4. **Pruning** (опционально) — удаляем лишние ветки для борьбы с overfitting

**Свойства:**
- ✅ **Eager learner** — строит модель при обучении
- ✅ Не требует нормализации
- ✅ Высокая интерпретируемость (читается как if-else)
- ❌ Склонен к **overfitting** при большой глубине
- ❌ Чувствителен к малейшим изменениям в данных


In [ ]:
# Decision tree на том же датасете
from sklearn.tree import DecisionTreeClassifier, plot_tree

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, depth in zip(axes, [1, 3, 10]):
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_demo, y_demo)

    xx, yy = np.meshgrid(np.linspace(X_demo[:,0].min()-1, X_demo[:,0].max()+1, 200),
                         np.linspace(X_demo[:,1].min()-1, X_demo[:,1].max()+1, 200))
    Z = dt.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X_demo[:,0], X_demo[:,1], c=y_demo, cmap='coolwarm',
               edgecolor='k', s=50)
    ax.set_title(f'Decision Tree, max_depth={depth}', fontsize=12, fontweight='bold')

plt.suptitle('Эффект max_depth: глубина 1 → underfit · глубина 10 → overfit',
             fontsize=13, y=1.03)
plt.tight_layout(); plt.show()

### 🆚 KNN vs Decision Trees — сравнительная таблица для IB

| Критерий | KNN | Decision Tree |
|---|---|---|
| Тип обучения | Lazy / Instance-based | Eager |
| Требует нормализации? | **Да** (использует расстояние) | Нет |
| Интерпретируемость | средняя | **высокая** (if-else) |
| Скорость обучения | мгновенно | средне |
| Скорость предсказания | **медленно** | быстро |
| Главный гиперпараметр | **k** (число соседей) | **max_depth** (глубина) |
| Главный риск | curse of dimensionality | overfitting |
| Подходит для | малые/средние датасеты | смешанные типы данных |

> 💎 **СЕКРЕТ #7 (Top tip from Baumgarten):**
> *"When choosing between KNN, decision trees and neural networks:*
> - *KNN suits moderate data sizes and low dimensions*
> - *Decision trees handle mixed data sizes well*
> - *Neural networks excel with large, complex data sets"*
>
> Это **готовая структура** для *Discuss [6]* вопросов.


## 📐 Часть 5 · Метрики оценки (часть A4.3.3)

### Confusion Matrix — фундамент всех метрик

> **Definition:** *A simple pictorial means of representing how well a machine learning model is performing.*

|  | Predicted **POSITIVE** | Predicted **NEGATIVE** |
|---|---|---|
| **Actually POSITIVE** | **TP** (true positive) ✅ | **FN** (false negative) ❌ |
| **Actually NEGATIVE** | **FP** (false positive) ❌ | **TN** (true negative) ✅ |

### 4 главные метрики (выучить формулы!)

| Метрика | Формула | Когда важна |
|---|---|---|
| **Accuracy** | $\frac{TP+TN}{TP+TN+FP+FN}$ | классы сбалансированы |
| **Precision** | $\frac{TP}{TP+FP}$ | FP дорого стоит (spam-фильтр: не пометить важное письмо как спам) |
| **Recall** | $\frac{TP}{TP+FN}$ | FN дорого стоит (медицина: не пропустить рак!) |
| **F1 Score** | $2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$ | нужен баланс / classes несбалансированы |


In [ ]:
# Worked example из MacKenty/Stephenson (стр. 275-277)
# Модель классифицирует email: спам / не спам
# Из 100 писем: 60 реально спам.
# Модель предсказала 50 как спам, из них 40 правильно.

TP = 40    # правильно предсказали спам
FN = 60 - 40  # пропустили спам = 20
FP = 50 - 40  # ложная тревога = 10
TN = 100 - TP - FN - FP  # = 30

print(f"TP (true positive)  = {TP}")
print(f"FN (false negative) = {FN}")
print(f"FP (false positive) = {FP}")
print(f"TN (true negative)  = {TN}")

accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * precision * recall / (precision + recall)

print(f"\nAccuracy  = ({TP}+{TN})/100 = {accuracy:.2f} ({accuracy*100:.0f}%)")
print(f"Precision = {TP}/{TP+FP}    = {precision:.2f} ({precision*100:.0f}%)")
print(f"Recall    = {TP}/{TP+FN}    = {recall:.2f} ({recall*100:.0f}%)")
print(f"F1 Score  = 2·{precision:.2f}·{recall:.2f}/({precision:.2f}+{recall:.2f}) = {f1:.2f}")

In [ ]:
# Визуализация confusion matrix
cm = np.array([[TP, FN], [FP, TN]])

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: Spam', 'Predicted: Not Spam'],
            yticklabels=['Actual: Spam', 'Actual: Not Spam'],
            cbar=False, ax=ax, annot_kws={'size': 16})
ax.set_title('Confusion Matrix — spam classifier', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### 🎯 Когда какая метрика важнее? (КЛЮЧЕВОЙ IB концепт!)

| Сценарий | Цена FP | Цена FN | Главная метрика |
|---|---|---|---|
| **Spam-фильтр** | важное письмо → спам (плохо) | спам в inbox (раздражает) | **Precision** |
| **Диагностика рака** | здоровому скажут "рак" (стресс) | больному скажут "здоров" (СМЕРТЬ) | **Recall** |
| **Идентификация лиц по фото для ареста** | арест невиновного (КАТАСТРОФА) | преступник на свободе (плохо) | **Precision** |
| **Обнаружение мошеннических транзакций** | заблокировать честного клиента | пропустить мошенника | **Баланс → F1** |

> 💎 **СЕКРЕТ #8 (ОЧЕНЬ ВАЖНО для экзамена!):**
> *"Explain why high Recall is more important than high Accuracy in medicine"* — типичный 4-балльный вопрос.
>
> **Структура ответа на 4/4:**
> 1. В медицине **false negative** = пропущенная болезнь = риск смерти/ухудшения
> 2. **Accuracy** может быть высокой, если болезнь редкая (модель всегда говорит "здоров" → высокая accuracy, но recall = 0%)
> 3. **Recall** напрямую отражает, какую долю больных модель находит
> 4. Поэтому **низкий recall = пропущенные диагнозы**, что недопустимо

> 🚨 **Common mistake (Baumgarten):**
> *"It is common for students to over-rely on accuracy and neglect the nuance provided by the other metrics."*
> — На экзамене никогда не пишите только про accuracy.


## ⚖️ Часть 6 · Overfitting и Underfitting

> **Overfitting:** *The model memorizes detail from the training data that is too fine-grained to generalize. Performs WELL on training, POORLY on test.*
>
> **Underfitting:** *The model is too simple, hasn't learned enough. Performs POORLY on BOTH training and test.*

### 🎯 Как распознать (Baumgarten)

| Признак | Overfitting | Underfitting |
|---|---|---|
| Performance на training data | **отлично** | **плохо** |
| Performance на test data | **плохо** | **плохо** |
| Сложность модели | слишком сложная (много фич, deep tree, k=1) | слишком простая (k=100, depth=1) |
| Решение | упростить модель | усложнить модель |

> 💎 **СЕКРЕТ #9:** на *"Describe ONE strategy to prevent overfitting in decision trees"* **[2]** есть несколько правильных ответов:
> 1. **Pruning** — обрезать ветки
> 2. **Limit max_depth** — ограничить глубину
> 3. **Min samples per leaf** — минимум объектов в листе
> 4. **Cross-validation** — оценивать на разных подвыборках


In [ ]:
# Демонстрация: training vs test accuracy в зависимости от K
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Сгенерируем чуть более сложный датасет
X2, y2 = make_blobs(n_samples=300, centers=2, cluster_std=2.5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X2, y2, test_size=0.3, random_state=42)

ks = range(1, 50, 2)
train_acc = []
test_acc = []

for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, knn.predict(X_train)))
    test_acc.append(accuracy_score(y_test, knn.predict(X_test)))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(ks, train_acc, 'o-', label='Training accuracy', color='steelblue')
ax.plot(ks, test_acc, 's-', label='Test accuracy', color='orange')
ax.axvspan(1, 5, alpha=0.2, color='red', label='Зона overfitting')
ax.axvspan(35, 50, alpha=0.2, color='blue', label='Зона underfitting')
ax.set_xlabel('K (число соседей)', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('KNN: разрыв между train и test accuracy = overfitting',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower left'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"k=1:  train={train_acc[0]:.3f}, test={test_acc[0]:.3f}  → разрыв = {train_acc[0]-test_acc[0]:.3f} (overfitting)")
print(f"k=49: train={train_acc[-1]:.3f}, test={test_acc[-1]:.3f}  → оба низкие (underfitting)")

## 🎚 Часть 7 · A4.3.3 Hyperparameter Tuning

> **Definition:** *Hyperparameters are values set BEFORE the learning process that guide the algorithm; hyperparameter tuning is the experimentation to find the optimal combination.*

**Примеры hyperparameters:**

| Алгоритм | Hyperparameter |
|---|---|
| KNN | **k** (число соседей) |
| Decision Tree | **max_depth**, min_samples_split, min_samples_leaf |
| Neural Network | **learning rate**, activation function, hidden layers count |
| K-means | **k** (число кластеров) |

> 💎 **СЕКРЕТ #10:** не путайте **parameter** и **hyperparameter**:
> - **Parameter** — *learned from data* (например, веса в нейросети)
> - **Hyperparameter** — *set before training* (например, learning rate)

### Методы тюнинга:
1. **Manual search** — перебираем сами
2. **Grid Search** — систематический перебор всех комбинаций
3. **Random Search** — случайные комбинации (быстрее grid)
4. **Cross-validation** — оценка на K разных подвыборках (gold standard для IB IA)


## 📝 Часть 8 · Mini Exam Practice (в классе)

### Вопрос 1 (Baumgarten Review #7, p. 38)
> *A maintenance system uses supervised learning to forecast equipment failures in an industrial plant.*
>
> **a)** *Define* "precision" and "recall" in this context. **[2]**
> **b)** *Explain* why F1 score is a better measure than accuracy when **false negatives have higher costs**. **[3]**
> **c)** *Describe* how a confusion matrix can be used to visually illustrate model success. **[3]**

> 💎 **Разбор (b) [3]:**
> 1. F1 score is the **harmonic mean** of precision and recall
> 2. Accuracy can be misleading in **imbalanced datasets**
> 3. When FN are costly (e.g., missing equipment failure → catastrophic), high **recall** is critical, and F1 incorporates recall directly

---

### Вопрос 2 (Baumgarten Review #4, p. 38)
> *A medical research institution develops a decision tree model to classify patients into risk categories for heart disease.*
>
> **a)** *Describe one advantage* of using decision trees for this classification. **[2]**
> **b)** *Describe one disadvantage*. **[2]**
> **c) i)** *Identify* one critical parameter that could impact performance. **[1]**
> **c) ii)** *Outline* its role. **[2]**

> 💎 **Подсказки:**
> - (a): высокая интерпретируемость — врач может увидеть **почему** модель приняла решение (важно для медицины)
> - (b): склонность к overfitting; чувствительность к малым изменениям в данных
> - (c.i): **max_depth** или min_samples_split
> - (c.ii): max_depth ограничивает глубину дерева; слишком большая → overfit; слишком малая → underfit

---

### Вопрос 3 (MacKenty/Stephenson End-of-topic #10, p. 315)
> **a)** *Describe* the role of feature selection. **[2]**
> **b)** *Outline* the three different feature selection strategies. **[6]**

> 💎 **Структура для (b) [6]:** по 2 балла на каждый метод (filter, wrapper, embedded) = что делает + плюс/минус.


## ✅ Чек-лист после Урока 3

К следующему уроку (семинару) вы должны:

- [ ] Знать **3 метода feature selection** (filter / wrapper / embedded) и trade-offs
- [ ] Объяснить **curse of dimensionality** в 4 пунктах
- [ ] Знать формулу linear regression Y = β₀ + β₁X + ε и значение каждого члена
- [ ] Уметь интерпретировать **R²**
- [ ] Понимать **KNN** алгоритм (3 шага) + важность нормализации
- [ ] Понимать **Decision Tree** + методы борьбы с overfitting
- [ ] Знать **4 формулы метрик** (Accuracy, Precision, Recall, F1) **наизусть**
- [ ] Уметь нарисовать **Confusion Matrix** и подписать TP/TN/FP/FN
- [ ] Объяснить, **когда какая метрика важнее** (медицина → recall, и т.д.)
- [ ] Различать **overfitting / underfitting** по симптомам

---

### 📚 Домашнее задание (см. `Week2_HW1_Theory.ipynb`)

1. Расчёт метрик **вручную** по заданной confusion matrix
2. Эссе: "Why high Recall is critical in medical diagnosis" — **в формате IB Discuss [6]**
3. Вопросы по feature selection и dimensionality reduction
4. IB exam practice вопросы

---

> 🎓 **Финальный секрет Урока 3:**
> На IB **80% баллов** на ML-вопросах идёт за **правильное использование терминов** и **правильную структуру** ответа по command term.
> Формулы метрик — **выучить дословно**. На экзамене вам могут дать confusion matrix и попросить рассчитать. Без формул вы потеряете все баллы.
